# G2-A：现有 G1 图像的 clean RGB 可读性先导

**选择 GPU runtime，然后从上到下运行。旧 `/content` 已丢失不影响：全部输入来自 Google Drive。**

本次只验证 RGB 能否读出实际 Token 的 16 组标签。零次新生成，不加载扩散模型；仅训练一个小型 U-Net。画质继续记录，未完成的人工评分不阻塞本次运行，也不会写成通过。

**预算固定：1 次训练，最多 2,000 步、batch 4、总墙钟 90 分钟。** 8 个已有源按固定 hash 分为 4 训练 / 2 开发 / 2 留出；训练只用训练源的 N/B0，共 8 张图，开发监测用 4 张。只使用最后一步模型，不挑最佳 checkpoint。训练权重、日志及结果持续保存到 Drive；再次运行会拒绝覆盖同一输出目录。

这是完整 G2 之前的小数据先导，**不是原纸面 256+64 源实验**。两留出源只隔离于训练，并非未被研究者看过的新确认集。尚不执行 JPEG、几何恢复、主水印或扩量。

### 1. 挂载 Drive、核对运行范围

不安装旧 CEG 环境、不替换 torch/CUDA；使用当前 Colab 的 torch、NumPy、Pillow、Pandas、Matplotlib，实际版本会落盘。缺少依赖或 GPU 就停止。输出目录固定，以防误点“全部运行”触发重复训练。

In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')
import torch, numpy, PIL, pandas, matplotlib
assert torch.cuda.is_available(), '请切换到 GPU runtime；本 notebook 不在 CPU 上训练。'
base = Path('/content/drive/MyDrive/CEG-WM/gds-v21')
source_dir = base / 'runs/g1-v21-20261128-8sources-48attempts'
output_dir = base / 'runs/g2a-clean-existing-g1-v1'
assert (source_dir / 'report.json').is_file(), f'缺少 G1 原始落盘目录：{source_dir}。review.zip 不能代替原图和 Token 轨迹。'
assert not output_dir.exists(), f'本次尝试已存在：{output_dir}。请回传已有报告/日志，不删目录重跑。'
payload = base64.b64decode('UEsDBBQAAAAIAAAAfF3aZsUU7wYAAKwQAAALAAAAZ2VvbWV0cnkucHmFV21v20YS/q5fMWfgTssexZCKYyRqFbSH5HpF2nxIAgMHgjBW5EokTHGJXTIVi/vxnZldSqTs9Azb0g7nfZ6ZWd7c3Pzb6D9UAz+/+7z6uo4SqAol65Udjjtdw0Hpo+rMEMFH3YFs4NPP/wI7NHlpdFP9oUx0c3Oz2Bt9hH3f5J3WtYXq2GrTQW36h1zmpVp4QiltWVe78dj0x3YAaaFpF4uPIXwI4befPn+ALSTx+jaE2/jNnfu/uI+R2rTRvq9rkSA1gRdAn0U3tGpLT2otu7vbYPHLu/cfv/zy5b8oIOIIOfEv5r9g8YmII0MI4k0U4uOYfgM8rq7OMR+T1+68APrxxNVIHQkxM87OKyZMxdavrpQjIVkT5zoKgsW7979++ekz+TiaXs+NrK4JMfOMRqYOxl4rst5dRTk7X0lGb648TPw5WCwWhdqDLAqjrFVWBBuWNfr3EHKECheoaooqx4fi5TqEl2sUYx7V9aYBcQvfgUABePECXgfwT5bj75GRX1UtRisHo/v2bEKbQhnUbxE1qhBGNgclPqCLj2rY1vK4KyTkGxDnWDzQIlvK9as7sVs6cK9Y7WZJhqNOP+yGDl1FpC131WEZBFFRHZTtBGrOvecHF5Y6tt0gPkzwVjWMNuZJ2cHMscrRPfg7InQa/sEHZ3Qnu0o3olAHo5T1UaJYrZwOfLA2sjgzTLWwCSMHkab4NddWsCD6vMKzrRp/zqalnfykM64QZkqybKxAh2HYvTZH0WqM1YZQej+7UnUyBJvLWoXQnfBvQLfLWaXJS+v8dPIBrCBZY0PAj5cEsKog+oKwYHVYGOLBj9Tpzbw3ucaRYZUwWAFVo4x3hZysWdeYfLJo0vUmI3iZNMlQ99meSeOMHDhzsjLmnnlPfCjunsZZOCpyhAQJ301MI5TqimAz5q7Ak2xydnfMWq5No4ydupmmcYjK8T+CFD/xP5/p01EwGdjfr7O5ezzqKMN11cj6EDVUpUu9vCV0mnL+DL3EqstTZbdJEB3l6eJ31Tw48I5Oe4M4SOAHrDBlBT/xJJsCojeOmDgizjImk0a5s6JM1xkZ4q8vsyAgrvUtmvrxvBgEMlvcIlsMN2AXrO5Nrh7GMTI6gksmpUWAHkmaBbmqa4wkuQPb79rqpGqbfY9+wlFhuKD7zuIa89qg0EdZNbypOCosS55fzas7HAPY0C7RdkB8n644aCP55/Qoz8eZlW5C3I6NIoTcImzs6fyEyCFsCEzRKxalVsFosTLCmG9qGJ7X4Ky7hnLeIdDyRyGwWYaxqqsk+P99qk11IPTQsvH6XH/ix9h7z7Qqbt1pozpVyK1J86gzjSLat9glc0qSMf9XWVcF2dUneLuFOIB/8PcfKC/uMMDbkT4wbAgeJFsVlPqXa8yT2/naCOHSqQfyDjeJtDSjBcZEI+DMhhauH1+N1d9LZZRg90KyhBM1GVvDqAZxJ/aVqotLUyOifq0aJQ1glVal7l5Yve+AubzI92cw+jSXWFjom4p60u+5MzKdoBsRfnp6i0/vOZyOvROhRdcq+Bsm9SN1hXePw5OVVXAv6169NwbzsHRGjr3tYKcg5btWcpctxwxb3rSzNuz6tqap8HzGBMlwJV2dGcmh84y2DTZ5deyPxIawwMUE9zFOHuxU4cfQuHNwzeJIfTJ/fPJpDw8Kr39BOrmFZFQOrP997LXYvqXr5bUSNzXFtyJzATivzgPR4sxUQu+sMl+5DS6lv57FFgOcMCJAJ9EQMml8jXrLodXYULayD62szPmiw4cQ1Cmv+0IRElIML3V9g4ABgzMaPl2qS7SSaOl0QwZML4jurpQ0PdLx3pttZheDEq24LLhYTk76RNJlMGPF4Wt504ol74klAbDB94LJ4gAcxQqW1BQPrZKPzDNZiCXvgdfP3028sK8gi16qiQMiuosdC0Fs7lu19+5tnqge0xnJtkUkCax8J8yWlvO2DL3Y1n0Ec62qRr3OaxenK9FTG0weDbDAuUNJ8ErqaVPyBRP+swHb6RZLqFnMVrVqunqAvKT0nt/GlrNGvEKNx9iur+riIddqv6/yCtX8Bcqe4tHfWVDYjSN808NLokB3BHPgsnGT5lujiTD0GPrcYd4Uvu0pQ+3g5C/JeHbimEsdioqdOLeTu9ZcmmvWDHgdIGsOjbNBSD+PSrWoCi3ixkXWjLt+xkKjt0CodIKjTx+zkPlJlEVCdojPGbY1GqGlTc09KwpLh9+oTacfVfPAA1Lw9zEf7jDfAJ7h/IbkYeXIs8kf4MJlyWYQXgybBhfp/2A8vuVX7OAvF0TVuBXNIm6Me8BN9lOH0BT3cUh2Ex/6uNm9LXSKjF0kU2bw70njKB9f9VIn5XmybDZlSXrxJ1BLAwQUAAAACAAAAHxdhFuosfcaAAA1TAAADQAAAGcyYV9yZWFkZXIucHmtXHtz28iR/1+fYuJUDkAWhEVZ0XnlYCtev26vdjcu7+6l6lgsFEgMSUQggGAAibSi7379mBkMwIed1KlsiQDm0dPTj1/3NPjs2bPvq67MZCaWhUxL8enD96KRaSYbUedF1YqqFCq9h+cfpvC52Efi50qsZSmbtM3hYdWItG3T5Z2Knj17dpFv66ppRdqs67RR0lwv1b35uEnVpsgX5pL/wI1oK9s0S9vUPPm7qkrzuVIXq6baijptsbPuJT7CpWnSpGVWbc2VytdlWpirNt9aUtomXcoFEGxufM7rVV7IC3Nddtt6L1Ilypon/fjDj2bCH7bpuh+papYbbkIfTaOydG5GZWnur7pyiTxLCxz9PTdaywoW3uxNo3VTdbUKRZpljVRKqouL159++kXEwvd+9kLhfX9Jv6f0+wp/f6Rfk3decHFxkcmVyPK1VK2PzApuLwT8NLLtmtLwPlKb9OpPNz6yj1tFuOfJYt9K5QdBtJE7PYYZ8qHJW5ngllCHUOBO6cHxBtDXj0Z3220NN/E6esjbTaK61SrfUYOIP4tvhBdBM892iHiaVu5aH+eKMtgL5eNcochBTMs2vgqFLFXXyCRVyzyP36eFksCvoqgekjIt+UbAY1YKVlYXsOM+DB8KJs9dEkimXlFTPSi9IiTYWU9U1bL0vQfgcykfiryUsecFuIkrbk99cLgGlgwjRm/zZfs3uuGvQrHKZZGV6VaquMiBqTjT7HKuaew78/I3pH7+8YfYlfqbVaiqa5YyUXWRt76+yDMQoGVVrvK1XhDcAcoUCJjMfCXdlkEo7uQ+LtLtIkuFuhW+nXgkLj4POfNosgS0Lc1Lbw7bqIJIlssqkyg8RnJCuM2LyFeiAAbiZOJ3sXjZM61JcyXF/6RFJ981TdX43rtdLZdApZC7dNkWeyHz9aZF68MkKy0tWqIfkWAPdBoowXly8WdxLSRIgPAyeW/v3eh7G9iJqmu9nrv4swIrlgO9IGMgXN0WjZskep80m4sK9IMMYbKeau6NeAzm72OT30NPUaQLmA6soBSqq2VznyvQ++dqWTV5uSY7+gpEqX2omjtYyVLm91KR7U2LqpRkSXFInsfoFl+hrqqquJdaQnCTJO7uI9CKN5Dc5UYu72CrG1mAmb6XQc9y1Eo9lHgubIPRqHrbdLsSHAGwpo7AqIMOKrT6eK+OcpWg8fSdCY7u68r7KVeKFt+IrlTpSsKIddfeikdDw5OzK7yqmXk0B6KNVesbaRmoL7REkAGNyXFEuGHKN3zw+GGEjzxt7cjKaAlFjYKeutUaFMQDrya3datA6WdzK8e+20S14A08kmnvw9TDpY0et53Sz5fVti5kK70Bn6AHagYpNDa7fjkepHe1iaEogYFRkT3dZTxiWu795hwJNMdg+KpJQAce0ibTjaeXl6QVDe47kRec09o3eugMyJl0ZU4K28h/dHkjMxD1SqRdC/aizZco79W9bPZ6t6VReJBgEJP1dPKY316+eAK+pwFrJtGQlmvpv+Q7Kd5Bt/hk9uXRb8AuGaPmzcGcz7y02XrzYLiOJ1ydmfPckn4rLWU87nMYDoZQYIQ9Kw+8VTMPlUB585ln/HlU78E0/s6KLWlwQsqSJAG7RHQJ/qDHeS5/7BYFMJAggtimdY36pFU0y1cr2YBHQlABZmadI84AqWRSyWSTC3DcxWzIsyGj5ta8sVUBHwz9Z/MLYzCppXYp2GHgR3ZglnejHcENi9CL7/ARb467YJKbWIyIApCQePDb7KdrnuAedmJG46cvWCHYVJgiRxyRt3uxzRWI5HLj2B20jsZoED0w/fMcUV9Ul2unIUEEgoOMDmrCA/l2SAF6n22k8s8SKfTBi4YCfgWogfBgCy6TVBNMvzfsecKKPiJRT7fkK2CgHfy3ehainikQsBLQsMRJR26uWS9geWUdpU2T7v18CzCu3dcyhltdXrYvR8uD22hG/QOG1Oznkra6AywWlfVnLzAQrM6Xd4XUKAx58nm4MMTiICvcFcj5PPPoVitrhVKC1/RsuNU++htDegIrTgtfD0V3yTx8+20oJlP8HwQH3ASWa3oBz9S8IWDmQrB1V9eB85h4Et2BpBrH5+Wdd2w8nBmMra8X82dxGYh/CnP5nbi+/PbmOCW6J7ecTabg4GLd/Iue1MhAXt6nRZ7p+OxXHAqjm7+D0aqsecUf1N0IzIUsMz8DZEobGeOvUFhVi8cGFLQtNkoXHqzB/pApien3yJ6gEV4vYvh/pju4MnBEMYc9fjDrOTJHSII75b8AzP/iKtCc/L34BeymIKXEzWHLh0KvEaJ4qLoiQxwkU3NT5IBuyI1GbLqxd2Kx00zjjcwalKHes23MaDpg5tyFtQin3eECcqAEeN275wz72w7Yt0TkiHpNvuRWqLaqwZfjpAodkV0JWFDcSkDccgiGOU7i+ULeGICvyyJVSnxfVMs7vyyjX0B30P6lhSYJ8WKS5LDsJIHFFBCwLHM0/g7JhGL9ILLtYKA3VXl/BcaB2sIGYXSVZeCS4ikYA2jwSf74G4QBp/feGaQ6P0hg1/GJoyN48lOVdYXswfevG4ne7zNoQQ1SAygBwm8ImZbAG/kKrSVxd/KduLnewb/pjfakRbXOW2VB9wE/zvGhVzJsGUkwPvzhCqSKef6ChDc0VyDLN9fjbi9Mt2vb7eYaLNPVS9sRPrPvGHbNTNfsynyY9lO/HA/yLU9vrr+9IeKGQ2IAin7C7g7SPIWW06CPMDRc1AIDOu5waYcUaH74+Kh/cmWfXPnvo/R+ndRVVcAcO+DclWMrdy9syxejllejlte25fWo5YthS6S7q/0dMOour0OxQB6M7S2rEj3yOYMDiunP3gNyAeRXoxWRNAT41xjHYXcym1zdnjaT6Ohjb5Fj7iBtENoW+bpMIB4E+K20s2SywGpm+TZ2PdgOFgh0878dbOnObvoL6LWzO49itptaOTiIlAbcsXvt76DbtUkowH5BaFQCsGP7NcwgMUcQZCaUJ+M20bKCjxD7g3ZsO+APUISONYhWACAgyoJIc3r1n9GfxERMIz0RW/6ba58/DOcB/wjQGuy333/idrClwL5druLLoP88NeST/0jqplooH3+DsOsslYY+isEPP3MAEFFqNBNMe+3ABGxI+nhz1ozb3IU1MDhLitve5uPMRT1wb+Th6A/MEW1lWvq0Lh9280VgFodxQr7khQGEajqb4DuyuiNrs1MiSJpqU7IfdqVRnb4g9sOefSxcMO7V3m4fmLQA9lIrtJASnkeAC/3AAT01AaVx1GrwXVEsi0pBx0h1Wx+9ANAKEWoRX8nJzWE3DcH2Fn3txXcxLu58LPWDBk/9Du2fU9qG0xJ6r+qGAlMEmOttugNyEH8AzAW8KtghUUAjFo1M7xQnrQRuNqq4qgg2LFJQNVB8RUNiUNVhOojZvsjLZdWVrT+9+eMeUEdNQH6bl8DSdbuJ0eJb5pMR1jwAV6TzHXZEzTFDuUJMFNuG34lLLX+qK1pKqAAUTJfLrkmX+5g1FSgi4eOFx2IfOA58kRYpgI4sOdGpJyTL0zUmugHQaULmYAI0JfaWO/ZS6sEmFHWsaVBQgS0sqZ71EJ8lDaRiD4ZyKidTMPA8uzuaniGhLZIqBiH29T1iEg7ASJMe6UEH+slsCnvuGh1M7yTYl7xqODGtpQy22KQxZtkQLaKuZDpr6hHQNwlLRHeZhtjYuk/xByOIyYN/TfrUzZrSNJgkICutbKSopVunKUkOFZ7h+AiA2cy6kNdMr9NgdaXy1gpwBYLta4hO0nnKpuJwRsP01E5W0oz5DQ0q9yjswYyazYezPo/tZ9pLtGV3UtbgNVX8a9NJPR0sdqRiPGnUQLxUYH661zI0uqlCsn2iOQBNnEb9ODAn/mXZGUBuHU7B4ra04g6M3ZAb0+f45wAZ4HgAxEGJkMQG4uwlUAAhtY9PQuEOEmgHeziK4URsPhw20SdJscNWe7jkB+Mga46+5i/s5vOSAisI57aU2yf5R8uAi8ZbRahjjkze50sDFehJJO9R/fkUpmvBpOoctREFqx4ObsTsTw8/QBbhypsHUVcq4JX8LP3LIAIO6ekGWRuaFaBMpKpVy7Z6djm3iGRKzhW6L+sOVs3gpR9Bk9gHgJhxrt2d1i20FcDVdYgEef36IUQvZBgQgYG23trcMshaiq6bg02wXOcZUUvZJIMzm91hpm5n+p0OsNB+7AaWx7FDaHxMHk4/4pzcgAxkg6Zm5ptPOouXDYginQnEH8himZZBn8DLjiTweAaam9n+pQTem6ZSaqIjYbXpVqtCihKYrQAjwMygQqJ9qGwqAONAXvBQWlyYeLj3DnAG40lGhI+XI9j7FNxCAvd98ByMhPncdHRaNjxg8h5RIp5uH7PZM5zk2fzpVtUphuET8DBtUxXekTO02e1LEBYIHNaeEw/oDsr4cCN0fHAfQ2SjOZMlmIA5tlZk/PxMbG4HACLBNddj5DgDBmjdoqyKT0m0ccqGjZedBKUWc96h+AfKrllGBDhxq8ZHSNtwAJUM9AXL6PioUX4TNO0wz2W31U12ZUeSXdnJZBfnuHrdCZGcTZXFvBq9kASZGpNIg2jjI/L17l54+hgSvPwf/7gdpQaNiRhmoRJSHhxt7mK9AVQZWBhzMEwVEMmqSNf6xJj5iyM7huWxz6EP0v86ve6CFnN2SpajOQ5bnrTzxVmHxwWUPIbGNH+/1W26KPCEE+lgpoIRADKOEaPpjDnX/9QrKSo0jgMOg1TAMTKww3p7UIjy0ueGu/nMW0rHkp42or6n/Tuu0XptvDA+Fz8Pdtm9JgVytXeRuhSBo+opOgDY/48Eap/vUkK7dCoxDIxLs/u0xDPOuOfipGHGnUnrHcYJzkjN0VVOeqacG5hlOl5UVeFrOiDsc4iDgODGiOcxZn7ncP8bbBwMSxSII3ys+i8VKazAlgitH+Ln599fspgPI34a3GgnYNCEzlXqioITKkHRmKqq23ybf5YNWKxW1uYgLuTQdJhQ5jIbrg9oG65XGRfWEJjDGf0eucWPd7f34M4A9m98jYpI1u7Ce1Q7BnF4fiwT6hUYM/3U75AlNLafBl2Y/thdRKzX0qMAZ02xexEy4ehqY14CmHy8SmgKFw0vuyx12uHlsHFCOQgM+rb1lyuD6qJqyVWa6hkuzQIDi0+weA3v9pdRp6TvvV6b88GD9lG9x094GlYXrdumBmGFu/gg483S9rDOuEoB65OAkueedoAR3DD7itHdsaYm7HPakvikCLCBAAhjFrxGuE15TP8lZgC5abqL8KFPo0S8d/y5AOwVcugWe7+a2HKBR6jizTs9E2BzNMfUIWuqukx9mE7JNp5hRU6CSjucCO7qafATtwGDVU08O9lbGJTU6s07PAkUvebo4BSMXODZYdPdBhMuvk4lUBi2rIqqib11I9E0Uj6m3Rcy9ibOPL+xIe1XA4Ph+c5OP/+rEXMSbOi31w8IlwqJhqXG8du8xbHfUFElO6Vb2IQd2AwblC+6DEGMnaiQazTEKzBPtCkvg1e4b1GLsXxSpHvYXF/fQ22Gv7zfMEmDIyZg5+75wAp4Wefx9PoS2uOOc07Nnubzjli3yS4o0kUCjqsPxH9wkwicfZQr8J8zdPbo6+c6R3FCtKbhVWjFa3oVGvGi+g2qFUAz8zmv/XQXimPmOiQD78DC34vX7MREDiojNmmTiXUH168EYpplkYMry1BG4DnmE7HiJW1Io1qMCVatDtAiOyaV5SFSMdAhdF1o70ERi8HajRedo8fAeof4COEM8mYUEc1GyGAMDHqkYkBSKFYNHXCWepsiOqZa7H0GQCOcnLIS9dkxWhHaOhoGxTfhcM3gqwCUYjmjZiEsYT7UNZyjhwgp9d+1+fJOHZuDRwGQxmFAfPUnPICMvQZl1huPw9riw6QglnyLFMX3/osF7tYN3kJ2q6T3nMCyuoKR/b+tLlq4Ext/kMKc1Bd9LNKVtuyv9wxNVbXG/fbFPbaUTtfL2bxvvh6Wp/nY/7m3viJQgoSeKlBzpj5ZDhjyfaRuVM3XV/KBcuMQIGr2NBlvuIV9+Jwqe/C5eXAODf2VYkqx7SD2XkBULlHnWsmlSLYCCWfZ3mU5YhQaknJ1IUCoXLVJdacPwXi1XOCG9ikHEd5WoMxVmS99+7jtbPSrGKGa2lSqycNYjxqBJHYl2kZvDEAYmlz2IMLYZVtnBw+x3jfpS/DoFjMmRrSltyC0BWUJwb74+mU/LFaq5O0+sfTIZQV2S9eAgviYVFJGKADz1ZnXd1fLqgZH8g65hE7jw1RX6qstYJoJ5ZdRWvQZBtXaAdG82GZLZIe2yjwU//3x3Qfc5fcfPzmz2Oq8BFaSlhhgmsJqoy2aTHziAdkJ8NU7qEB91deJvWIICZ8tNoTPP1elfCWqIgOQV2aFvUcDOUXmpKhu0abZUAN99q5beJ/mxWSFuZ++OAJhebUS7YbKVyXEDHfitx8oOKCTAPCUfZ0c4U69kN43UC098JawBMYe3VZb1aNFOr+CrALdWiu+vZxAuNmBHjxsqkJO/l4ttMcHadmkoCy6MhK0XBLHHXM5ZBCHPRH/8fXVLz98eP3jp59CS6Fzis8twC81W1u0DZEm7lmiQP7KTLm5E8xWVPp9BgbPuUrSe2Aqus7jtb2fuhJ1Uy/2FwJh4JM/fPwNTSWpLb4poorqQbzBm8ByLMiYWPizAvnF9zAiZ906VAG/ALxO2g1KtvKdAgWdhsNkWl+ODhewnFdOmu74cx59m5agksmxJiM6iBVOc4oiznfBBYH4KexbltECYMRmmzZ3sIukT+ebQzgmG5Aa1PUldEELeaYHEtduO9xprLxrVy+uvm6eM+1Zy6KuzjDGWtfdOKbi3DvXrl6ezCDey0aRwXysbw9f8Yn0Yzw9RthTcxaJZgKIRMl5+PsxRzoRhVGUBB/6kMpz49DhDyZRTUD5WEdI6a0tHu/nQ9+LMW2DPinQR/78YgpRExEN2vpAPL1rbXoLf44UWwEzz74nYPuODZ0b9hpzp6P8k33cSl5rIvHidBcXYxwSxW1qgHIVxEq8XveFHA1WPly9Tj69e/32p3e6yQFWwR8AuGuyXbPHu9tsdsd5rDvmLMHY0EnAhpRLDHW2MXgS/xSPXu9wvFvrcZ6OFeIdWyvOkTAVZsF8FZwQdZefXM5o8i1Ujxcj7NV7ctbfD3/I2iWMXuOXFOGaq+vet+o7U8fQcZxCGcLReXMfutSEaz8nGObQeV42jP8TBBm6LxXpYtaZL50kIMnt+TNriHJOnFgba8jrhNvgWOaOA73/mrHpRZ1/beQdneGQs+CD62Pnhv20sEjHVO9tZ6eQqT8Dt4VJ7kHDaCyMocq1c4DIZtEOzJe+h2bT8W8GEukqxqOHmS5c4sHoTvQ6S7d/4+QdonKwamC7FML+orHw16YCEIEjDH6gSgCw2st0b1u5N10fNtSIfpIYz71rPDTFg/PehB4SE4RHcPT0tEkaQDwX4ZHy9J4H4UOb1DK9S7ZyWzWMp5Vbf5niKyg90z6YN1uAy6ddPp1Z9tRh1fuxHNrpNwCLaj2ER6feA4SGgzcBMVTBFJKHWTX4o3NjofcA3vkYSusHH74n+AppiFZFpzb+sDXuE07Svz8zDYWLB3XR/TfTEcSzohoRG0bD4g8fTjOnEXPhUSBlBkA7glDY14pivS3B7NZMTPnChF5PcBT6QPoj+F8l6yalFxaTtkpKiBOcAg/3h0t4+yqAGdA3d5TrWA+F7d9HS8zcJTpz5+taYF22GIr9l0caoGdb94bjH+Eq/jCAfo9TgIB9xCymhtE/I4uwO5HnHaeawBzV3h6lhSrsIuSbyYJhfEXm9QiZ3NLU6J1T7H9/LThDDgw+sh732EDWRxbEJkHrCh6A4odjyyY5j2MxFUbo/yCuLvHO5XHKWd9gRO/w1Q53zD9YjcEekt5cm58ZF39M2eCoVEbe20qZk10tWcPSupk5/x7Wt8rMrVoIXYc1u3SPONn5yvv5EQHGH52VsCkVc2qDAqcL80j2hgdGMCORG+uk/ZmjO/PjWrd4nN2Z6MTPcRpH70H7TPMZ+9dvBx6xOS+U655gj7HXCZtyKAD9qcNXyMH4hI8z9eBXVRvVAH9ZKPqjPmK3zk2Nsf9g3IGP/ip2fs2+kGvFuBBfQ9FfBRAPwuCd8by2lX9CmP4lJ/978R5TMBPitXOuYxJkYgEcmyCSrOwR68RWW/Yd+rRNX252rlbtyzCa0u7JoIQ7ubm2QFpP4+QmxoVeXyoSO5VK6L+Z4ODkT39NwZeJt4UiPfI3lI0KAGyCya0VORUk6SSmfZX4eH7QeewOHdPvoTQCBqWX42PAypjXTHSHqkzaTa4SSnMm9F0kSQM7Jx+ShQSLJhOT06R3/NF/rYCT1FnbvBWhdTrRP6EAXIwzmBcDTRC4dFHkoIMY4AHIcknBvUw4kZfgK1oJeFZNmTeqdQJOUq0rxZ23eDlzo1p0Z0feC/83Itn+wJpuyd1S1q34PlXyHX3EEirAqhI98+0XthbXVGMKmpvHK++R6mLpCt98QviaJE+34pFuPXmnzIvdWg8LcHmZzLKyAo7B1uEHm3B24AEvfJXmRdfIw0yE/QqXCA9rU9iw3dK1RWzvj7c6tPkrND+Fk08epE+dFxVAzNy8rIOubr+QonW6DYZjOz/0H8sNInWqI6cSsr5oboPCpJNEuhbDgE+dvXiODYP+SxnQZOrMl9vgd/FmCLwNyjL5cE55gOxrYkhU6UCVr4921s8SSpoo6nKseb4ytw/95hck0hUoTMzgG1nrBg9XtKiQdHlj4zUbhVRHjpWMp/y6OFUc+rAfEXaZrzagA2fHlTUSvylF4XcpvW3ye/mKD28EWw1+T/EeX1G1Z2e9M6OQVH9FUfS/ef0eNxbPnlDbv/G05YmgBYaOXmib/vAxefvu/Y+vf3339thr2AT18Yt7+pf48YTO5kOP4H3MkOK39/TiRYcpzhf64KvsCG5stEEPUeYizD7jWnxdmXQcNH1mNdflULa3eaVlBON4G8IjXwdkvivoiLrrYizue3FxkePbnmzTqPQhSXCzksQzX27UKArnzVdaRa+bdbeF0T/SEw05uVmUZlmS6ue+N9GFyl5oXrxw6DjRg0HC0R7QjEANd6Q/2FXZNxL0ez14VI33I51+ps88rj0O5sYz8wUh8+EXhIxPe3/ZAzrbvtvlLZ7F/B9QSwMEFAAAAAgAAAB8XTkC7NtOAQAAQAIAAA8AAABnMmFfY29uZmlnLmpzb25tkUtLAzEUhff9FUPWnZKEKqWuWi1DQbqwLgSREDPXTHAmGZNMH4r/3TxGBXF7vnPvybn5mBQFcp5LQMsCVXTFrm83qx3bPGz399tdxSqCptFjoQXukkvWrjxQUkrKS4rpJSF0kU0OoA4OuogSxSSLfas8q03HlU4hN/swPiNlSCudGayAMnnKA1nmPd4GL8uMCTNoHwbnCdVw+AtoAg20tRn8/zAv5LZzQXpEOzQt0BqjpwQ7fmLOQx8ZxRgn8Zl70TCn3uEnOhzAaqUls9xHFc/wWPEISjahIwh+HsFI4m47hC4gjK5jwsV8TBANiNfeKO0ZHMDGQTKiWPJbo6Om4cgkaAjhyui4Cf9Wi6/i3kPX+0hy9tvAw1XPTDkm84tfeOsgMT7IDrRPu+KnaKPhqhChoi7uqnVhdHtGo9OHr/NKMDj1XLs8kDZNPidfUEsDBBQAAAAIAAAAfF2cUpwomwkAABUTAAAOAAAARzJBX1JFQURNRS50eHSNWF1z27oRfeev2Dc7U1KUFNd17SfHSdzcSWzf2Glm+tKBSEhCTRK8AChZ+fU9uwBp5WY605kkZvix2I9zzu769v0j7ZazBd0ui+tL0i/GB9NtCtOqjaaq0aojp1WtVqYx4UC9aWzIaTlfnheLRbG8yLLrIWytMz9UMLYj1equxt9wSWGrafDawWrfmMqE5kCt3emaovU/BiU21862hHPwkTtkGxU0BUuKKtvBl8EOHh86XVlX41u9U80QzzJBtzm+rPH0j0H7gMfpI1y1Gn7VGV43dXy/1o3ZaYdna+tG13Q1RGMd3dhGrWb04PTO4FS6XcCwH5rg5ZBe9dplvbPBVrbBo1bho6Grtqrb6PqKOkut8R75m2JzitNJxlOr3LNmI97repZlT1vcxB9Fdx++5+Rb1TRwqHdwsjWdcgdaOQXjtNJwV0s610PTRD9QsBl9CmyhsyHjh33yG9mCk73FObT863nh7eAqfO/gLjtT0vnZeLPWO93YngvGZdLO8OUsg+UKdQ/a0w/tLHV6H6vmc2qsqvlUqs16PXgkr/zn7+WTG3z4giBRYyTax8IgyZ5s1xzEwxFedHbByf16+47WptExvXjBOFJVQOpwu8O/T/ZZd4SiwN6MHqPLHmDiuLPHf1wjvNPaSh3+QjGkf5v6TY7vnQ90FoPO4f5LoOVxtAhDeb631U1NdkDM18itNy+kXAuf16hMSpIP6gBIbgAo7Wb0hEjgfUwH7QGo49TrF4Xy6TrPvKWT0foJ4Kg6P53G9plXKOTaBM5JTnf3TzhzDchtyVcGTpq1qRjRiKYVCAM390gm3ZXv5pE3ggq4+Vre6LRnOr1ap9OL5PCbGb1/TUMmBToyt7dkOczJzOnZ+GEutGltZwL4Dptc1xndd/DAvABs34o7HTIgUnxJNFMBn3Cm5/M5Xdeq/Y586h74WKkAdJ+hEo5m8/kip33NF3zlNYP3YrFYXizx/2wN0IW3y5xhp4YNux4TQteu2kIIABvHmWgau/eSlUiTyATVXNLbZXl+VsJgCdRkCLRAZne2ifRfNbZ6hlNh67QmBZlAzEVvbcORNpwxdhkq2GnlgKTKcvH2JqBYz6bPYKyDF7CF9xbnYMnGsHBES9C8DqJ2fvZyfkaVbho/0trbdWjVi8AK/xU+wxjEDhQIuoTG9Q1fAJYlomn7UEZemK4HbEkAwQEnPL3mn1VGOcAWbsD05vRXcqGooD3ox+l+WbLwsqnkp8j/xpk6n0TG1Fqx9q3hbAduqLoGYMHOz9b7S4puQ7sY7pAQ3BRdtz2QcmfpofC9rhjX2YT7MAhu1ZrxygRhfpiOX+R05lRtdfXcW2QQuGjGu2oIlllRwRs0jjxjrW51Dh8QU9CdF3x8lGAZcgmv1/T3eQGKDugz+61tdPEfu6I95JdYeXFGsL3PpLKwHJF85AJr1ae7m/svD58/PH1gcQiDv6Kbb++v6f7+S9kxX8ERiKsKqlwrKFydSbGiGS8nsCjtmGd43AXpcPWAhIp6cgyRHt780DO6mY5HOvHVIVuATcIj8Qdg86XTvXVBSu0Vt1kU871Dx0Pmkaedjg0Ot0EZhsnAHQKfZ7DIyfLl7cM3+X7sttLTTAdkGm6ut3PiN4ABaBN3N7Rdgz7I3YO5PmoQgHct1eQmDvhqJMMNXDMkGdqPgidMepwTnYgy37DbbKWEVpcJDLNswjjr0wiRUaFAih1cilJoQmGA5h5TCHtVG7XpID9AyeliObYwVKNqhlpQdyT6kMYvgJKpom/s+sBhQ8LY9tvly9tl6klMClodEr1hKJuEGHKD0A3SEXnOidEKGZcvZ/Q1lamqBqeqQ06QqTRvrVSDjg+0jA/pFJ3FcyBVw4ND7KV5kp2hZ0ug8E88y6VjSF/G1QDI5BnnN6arECmT2JQzHkUHOrQRyV/Ml2fRSxYh99oM4yjAXDfMPdVkxzlOdRB+Q32Ljw9fIeAHjC4OvSLOhcJKFYKqngtnVxgVOqgGR2VaoOURE2SomCCWg2igJMAIMtdeTaAqOHQk3g59sZaBr6t40kGm//RShhKIo/EpnX5WkFBwa/HmKgpYkWSLG3dtRFDo9AgrbAXvRkXNXisqDa07bpAnYDNm3gJ5knmGK4NphNHA9+O0coUfSAOwdsj8dlivGeZJrl+NS1GncfIIFmO0hpOc4HOXeY0KoiyMh5WuFNiMiZgq5RxeFOK2cQgcgx1biYy4KeyIT7ibSQ+fEpMKwRrBpoIb9J/GsW/dc2f3XSHIHMEYmcPNYtI1PDUb1lAZJKEsSINgZejQ5bvxxKJRB2b10Xwv4yLPTE5fgV+goQh0Gv79tFfwlM0VwStccqhNJ9zmImBS2UgHwMS2VWDBACayMAN1H539obufRm52lTtkozaXr8zFsDbRe1QfmZlEyPIRJzcfqAWyUQ24mvBbTlAtJ1QyG1eWGRyxMKE+zsy/qsD/Y7WcOvEjS1igDs0QF4gGqwdykC10cR43jmBYjxstbZ3n3NdpU6/XaLAFt53ETuK5OCqbisuNSSP9SRwQUs4E9Xgkm0xcFJlBO6P3J+mgqWNj23wHWMq2JENcxm1yiPPPSTLIn2CpU6vG+C3wM3S1FHk8o+Ay0mqoMeGkE9AKpFYqZOwEEzLWJuJSCOHNqmEKcM8D9SBgseGMhd0oNNXWomGy5iVRYIUHn9HKeDJB1QynDKVhmUzi5aMPWyPgSwF5fsNxGE7m8KMWeROzd2hXsk0qOb22wt2jyGmjbSttSdoyd/8ZfWc/2HmRY7Csthl/tVeuH0UNKYvGETxX3Q8riFvguWc9qTH7hDRF5NVWjkarFkX42zypZST+Uc9gg/3gmPE4BW7i45hsWZdndDv57DEuV9s8++3hw21OvKoVeyTW8T4sMWxc6hBx84ky4Muxu6fV8OfVN0vL96iBxLImU7mRQWZGv48LuEwy/pdtXeyqHYqkGA5Z9okHtMs4MNGm9sVuuSiRCV9uFnxd8O88eB8pLpJrxdkFcghNRcDZ/RD+5/dLVQhTiunXK2xyQacWq1MywVRZyzLmNH68mWVJ6P/16UH0Ad77uAtuhjFjcajcKtAD4JNWw+u592VMPm6+NjiMAdNgENN61PB6sBVDLKfRh1kfUr48vXYZcH4cJ++TqGa8BrMPkV7iSJ0WmzLO83xOHHQx2EJhsJWw9qMxC73iQFaGOHTASKbbla5rWZjiGjeM/Ac09craZ941WNbY1TRFYofa8wV6WIikgz5dASV4LQ2943oMJwXkK/QeQfoqTpasbaAc//Ljv1BLAQIUAxQAAAAIAAAAfF3aZsUU7wYAAKwQAAALAAAAAAAAAAAAAACAAQAAAABnZW9tZXRyeS5weVBLAQIUAxQAAAAIAAAAfF2EW6ix9xoAADVMAAANAAAAAAAAAAAAAACAARgHAABnMmFfcmVhZGVyLnB5UEsBAhQDFAAAAAgAAAB8XTkC7NtOAQAAQAIAAA8AAAAAAAAAAAAAAIABOiIAAGcyYV9jb25maWcuanNvblBLAQIUAxQAAAAIAAAAfF2cUpwomwkAABUTAAAOAAAAAAAAAAAAAACAAbUjAABHMkFfUkVBRE1FLnR4dFBLBQYAAAAABAAEAO0AAAB8LQAAAAA=')
assert hashlib.sha256(payload).hexdigest() == '7a09bfa952c72c2dcb1fe69a30cad74e6ad28c74d21e57e6e153db059b2a63cc'
task_dir = Path('/content/g2a-tools-7a09bfa952c7'); task_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for member in archive.infolist():
        assert Path(member.filename).name == member.filename
        (task_dir / member.filename).write_bytes(archive.read(member))
sys.path.insert(0, str(task_dir))
from g2a_reader import source_split
config = json.loads((task_dir/'g2a_config.json').read_text())
split = source_split([f'g1-{i:03}' for i in range(8)], config)
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)
print('源级划分：', json.dumps(split, ensure_ascii=False, indent=2))
print('训练预算：', config['max_steps'], '步；', config['max_run_seconds']/60, '分钟上限')
print('输入（只读）：', source_dir)
print('输出（Drive）：', output_dir)


### 2. 执行一次有限训练与留出评估

标签来自实际最终 Token，不是理想空间模板。所有同源方法都留在同一集合；P/B1/B2/P−E 不参加训练。每 100 步保存 `latest.pt`，每 200 步记录开发交叉熵，只监测、不选模型。

出现输入损坏、OOM、非有限值或预算耗尽就停止并保留记录，不自动缩模型、重试或追加训练。正常完成后在全部 48 图上输出分集合诊断，**只有 2 个留出源的 12 张图属于未参与拟合的评估**。

浏览器断开不保证后台继续执行；如果 runtime 被回收，Drive 留下的报告/检查点可供审查，本版不自动恢复训练。

In [ ]:
log_dir = base / 'delivery-logs'; log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / 'g2a-clean-existing-g1-v1.log'
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'; env['MPLBACKEND'] = 'Agg'
command = [sys.executable, str(task_dir/'g2a_reader.py'),
           '--source', str(source_dir), '--output', str(output_dir)]
print('Running: g2a_clean_reader | log:', log_path, flush=True)
with log_path.open('x') as log:
    proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    try:
        for line in proc.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        code = proc.wait()
    except BaseException:
        proc.send_signal(2)
        try: proc.wait(timeout=15)
        except subprocess.TimeoutExpired: proc.kill(); proc.wait()
        raise
report = json.loads((output_dir/'report.json').read_text()) if (output_dir/'report.json').exists() else {}
print(json.dumps(report, ensure_ascii=False, indent=2))
if code:
    raise RuntimeError(f'本次停止，不补跑。回传 Drive report.json / failure.txt / 日志：{log_path}')


### 3. 看留出可读性，而不只看训练拟合

`reader` 必须优于训练集组频先验、位置先验，以及使用另一源 RGB 输出、打乱空间输出的控制。N 单独报告，因为它没有目标地址偏置。`address` 是理想地址猜测，用于揭示只背空间模板的捷径。

balanced accuracy 为每张图实际出现类别的平均召回率；交叉熵越低越好。类别支持、混淆矩阵和逐图结果均保存。不要把 1,024 个 Token 或六个方法臂当成独立源。

In [ ]:
import pandas as pd
from IPython.display import display, Image as DisplayImage
table = pd.read_csv(output_dir/'metrics.csv')
heldout = table[table.split.eq('heldout')]
display(heldout[heldout.arm.isin(['N','B0'])].pivot_table(
    index=['source_id','arm','method'], values=['accuracy','balanced_accuracy','ce']))
display(heldout[heldout.method.eq('reader')][['source_id','arm','accuracy','balanced_accuracy','ce','present_classes']])
display(DisplayImage(filename=str(output_dir/'learning_curve.png')))
display(DisplayImage(filename=str(output_dir/'heldout_controls.png')))
display(pd.DataFrame(report['signal_flags']))
print('诊断：', report['decision'])


### 4. 到此停止并回传

预先定义的“小样本信号”标记要求四个留出 N/B0 图分别优于指定控制；它只是下一轮审查线索，不是统计显著性或发表门槛。训练好但留出差，或不优于捷径控制：本小数据预算尚未建立可读性，停止审查，不自动扩大。即使全部满足，也先审查读出场和证据，再决定是否进入真实 RGB 几何恢复。

本轮没有未见地址排列数据，故不能声称已完整排除布局依赖。失败也不能证明更充分数据下的读取器不可能。原 G1 结果与画质记录不变；本结果不证明画质可接受、盲同步成功或高鲁棒水印成立。

In [ ]:
print('回传压缩包（Drive）：', str(output_dir) + '_review.zip')
print('读取器检查点（Drive，单独保留）：', output_dir/'latest.pt')
print('原始输入未变：', report.get('original_files_unchanged'))
print('到此停止：不自动开始几何攻击、扩量或下一次训练。')
